# Assignment 3: Recurrent Neural Network (RNN) {-}

This assignment aims at familiarizing you with training and testing the RNN neural network for an image classification task. You will go through the process of loading data, preprocessing images, building the RNN model, and evaluating its performance.

The assignment rqeuirements include:
- **3.1 (1 point)** Load the dataset.
- **3.2 (1 point)** Analyze and process the dataset.
- **3.3 (2 points)** Construct an RNN model using GRU (Gated Recurrent Unit) instead of LSTM, as demonstrated in the demo code (https://www.tensorflow.org/api_docs/python/tf/keras/layers/GRU). Train and evaluate the model’s performance on the test set.
- **3.4 (2 points)** Construct an RNN model using Bidirectional LSTM (BiLSTM, https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional). Train and evaluate the model’s performance on the test set.
- **3.5 (2 points)** Construct an RNN model using Bidirectional GRU (BiGRU, https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional). Train and evaluate the model’s performance on the test set.
- **3.6 (2 points)** Compare the accuracy and runtime efficiency among LSTM, GRU, BiLSTM, and BiGRU models. Provide comments and observations on the performance of each model variant.

The dataset you will be working on is imdb_reviews. This dataset is a large movie review dataset. This dataset is for binary sentiment classification containing a set of 25,000 highly polar movie reviews for training, and 25,000 for testing. All the reviews have either a positive or negative sentiment. Reference: http://ai.stanford.edu/~amaas/data/sentiment/

Each data sample contains:
- label (tf.int64)
- text (tf.string)

### Submission {-}
The structure of submission folder should be organized as follows:

- ./\<StudentID>-assignment3-notebook.ipynb: Jupyter notebook containing source code.

The submission folder is named DL4AI-\<StudentID>-Assignment3 (e.g., DL4AI-2012345-Assigment3) and then compressed with the same name.

### Evaluation {-}
Assignment evaluation will be conducted on how you accomplish the assignment requirements. The model accuracy on the test set is one of the most important evaluation criteria. Therefore try to push it as high as possible. In addition, your code should conform to a Python coding convention such as PEP-8.

### Deadline {-}
Please visit Canvas for details.

In [1]:
# Note: to enable GPU training in Colab, go to Runtime > Change runtime type > Hardware acceleration > Choose GPU from the drop-down list.

# Download tensorflow datasets
!pip install tensorflow_datasets

# Import libraries
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf
import matplotlib.pyplot as plt

#3.1 (1 point) Load the dataset.

In [2]:
# Load the IMDB movie review dataset, return text (movie review) and label (positive/negative)
train_dataset, val_dataset, test_dataset = tfds.load(name="imdb_reviews", split=('train[:80%]', 'train[80%:]', 'test'), as_supervised=True)

print("Training set: ", len(train_dataset), "samples")
print("Validation set: ", len(val_dataset), "samples")
print("Test set: ", len(test_dataset), "samples")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/25000 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.XVV0MM_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...:   0%|          | 0/25000 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.XVV0MM_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...:   0%|          | 0/50000 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.XVV0MM_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.
Training set:  20000 samples
Validation set:  5000 samples
Test set:  25000 samples


In [3]:
# Show same samples in the training set
for example, label in train_dataset.take(3):
  print('text: ', example.numpy())
  print('label: ', label.numpy())

text:  b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it."
label:  0
text:  b'I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film was rubbish. 

#3.2 (1 point) Analyze and process the dataset.


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define parameters
vocab_size = 10000  # Use top 10,000 words
max_length = 200    # Max length of each review
embedding_dim = 64  # Embedding dimension

# Prepare tokenizer
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts([x.numpy().decode('utf-8') for x, y in train_dataset])

# Tokenize and pad sequences for each dataset
def preprocess_data(dataset):
    texts = [x.numpy().decode('utf-8') for x, y in dataset]
    sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')
    labels = [y.numpy() for x, y in dataset]
    return padded_sequences, labels

X_train, y_train = preprocess_data(train_dataset)
X_val, y_val = preprocess_data(val_dataset)
X_test, y_test = preprocess_data(test_dataset)


#3.3 (2 points) Construct an RNN model using GRU (Gated Recurrent Unit) instead of LSTM, as demonstrated in the demo code (https://www.tensorflow.org/api_docs/python/tf/keras/layers/GRU). Train and evaluate the model’s performance on the test set.

In [5]:
import numpy as np

y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

In [ ]:
print("Shape of X_val:", X_val.shape)
print("Shape of y_val:", len(y_val))

Shape of X_val: (5000, 200)
Shape of y_val: 5000


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

# Construct GRU model
model_gru = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    GRU(64),
    Dense(1, activation='sigmoid')
])

# Compile the model
model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model_gru.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val))

# Evaluate on test set
loss, accuracy = model_gru.evaluate(X_test, y_test)
print(f"GRU Model Accuracy on Test Set: {accuracy}")


Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.5316 - loss: 0.6836 - val_accuracy: 0.6944 - val_loss: 0.5942
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.8171 - loss: 0.4221 - val_accuracy: 0.8854 - val_loss: 0.2759
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9355 - loss: 0.1817 - val_accuracy: 0.8894 - val_loss: 0.2644
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.9670 - loss: 0.1040 - val_accuracy: 0.8812 - val_loss: 0.3225
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.9822 - loss: 0.0658 - val_accuracy: 0.8762 - val_loss: 0.3883
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9886 - loss: 0.0460 - val_accuracy: 0.8738 - val_loss: 0.4536
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9922 - loss: 0.0287 - val_accuracy: 0.8684 - val_loss: 0.5309
Epoch 8/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9956 - loss: 0.0182 - val_ac

The GRU model achieved an accuracy of approximately **85.22%** on the test set. This indicates that the GRU-based RNN model is performing well in classifying the sentiment of the IMDB movie reviews, with 85% of the predictions being correct.

#3.4 (2 points) Construct an RNN model using Bidirectional LSTM (BiLSTM, https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional). Train and evaluate the model’s performance on the test set.

In [ ]:
from tensorflow.keras.layers import Bidirectional, LSTM

# Construct Bidirectional LSTM model
model_bilstm = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    Bidirectional(LSTM(64)),
    Dense(1, activation='sigmoid')
])

# Compile the model
model_bilstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model_bilstm.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val))

# Evaluate on test set
loss, accuracy = model_bilstm.evaluate(X_test, y_test)
print(f"Bidirectional LSTM Model Accuracy on Test Set: {accuracy}")


Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.6671 - loss: 0.5916 - val_accuracy: 0.5850 - val_loss: 0.6799
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.8260 - loss: 0.4202 - val_accuracy: 0.8500 - val_loss: 0.3427
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9072 - loss: 0.2522 - val_accuracy: 0.8702 - val_loss: 0.3408
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9449 - loss: 0.1599 - val_accuracy: 0.8546 - val_loss: 0.3507
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9598 - loss: 0.1205 - val_accuracy: 0.8600 - val_loss: 0.4221
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9666 - loss: 0.1024 - val_accuracy: 0.8542 - val_loss: 0.4401
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.9786 - loss: 0.0675 - val_accuracy: 0.8470 - val_loss: 0.5046
Epoch 8/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9688 - loss: 0.0936 - 

The BiLSTM model achieved an accuracy of approximately **83.88%** on the test set. This indicates that the GRU-based RNN model is performing well in classifying the sentiment of the IMDB movie reviews, with 83.88% of the predictions being correct.

#3.5 (2 points) Construct an RNN model using Bidirectional GRU (BiGRU, https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional). Train and evaluate the model’s performance on the test set.

In [ ]:
# Construct Bidirectional GRU model
model_bigru = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    Bidirectional(GRU(64)),
    Dense(1, activation='sigmoid')
])

# Compile the model
model_bigru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model_bigru.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val))

# Evaluate on test set
loss, accuracy = model_bigru.evaluate(X_test, y_test)
print(f"Bidirectional GRU Model Accuracy on Test Set: {accuracy}")


Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.6274 - loss: 0.6105 - val_accuracy: 0.8242 - val_loss: 0.4294
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.8824 - loss: 0.3015 - val_accuracy: 0.8590 - val_loss: 0.3280
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9300 - loss: 0.1946 - val_accuracy: 0.8730 - val_loss: 0.3290
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9612 - loss: 0.1140 - val_accuracy: 0.8650 - val_loss: 0.3728
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.9802 - loss: 0.0644 - val_accuracy: 0.8590 - val_loss: 0.4721
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9892 - loss: 0.0376 - val_accuracy: 0.8646 - val_loss: 0.5137
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.9943 - loss: 0.0226 - val_accuracy: 0.8510 - val_loss: 0.5911
Epoch 8/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9911 - loss: 0.0292 - 

The BiGRU model achieved an accuracy of approximately **85.1%** on the test set. This indicates that the GRU-based RNN model is performing well in classifying the sentiment of the IMDB movie reviews, with 85.1% of the predictions being correct.

#3.6 (2 points) Compare the accuracy and runtime efficiency among LSTM, GRU, BiLSTM, and BiGRU models. Provide comments and observations on the performance of each model variant.

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Construct the LSTM model
model_lstm = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    LSTM(64),  # LSTM layer with 64 units
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
print("Training LSTM model...")
history_lstm = model_lstm.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val))

# Evaluate the model on the test set
loss_lstm, accuracy_lstm = model_lstm.evaluate(X_test, y_test)
print(f"LSTM Model Accuracy on Test Set: {accuracy_lstm}")

Training LSTM model...
Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.5530 - loss: 0.6729 - val_accuracy: 0.8090 - val_loss: 0.4743
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.6560 - loss: 0.5932 - val_accuracy: 0.6546 - val_loss: 0.7270
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.7464 - loss: 0.5313 - val_accuracy: 0.8018 - val_loss: 0.4705
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8345 - loss: 0.4050 - val_accuracy: 0.8246 - val_loss: 0.4307
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.8255 - loss: 0.3998 - val_accuracy: 0.8448 - val_loss: 0.4070
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.8824 - loss: 0.3036 - val_accuracy: 0.8676 - val_loss: 0.3191
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9357 - loss: 0.1829 - val_accuracy: 0.8676 - val_loss: 0.3382
Epoch 8/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9604 - l

In term of testing accuracy:
- LSTM:84.09%
- Gru: 85.22%
- BiLSTM: 83.88%
- BiGru: 85.1%

In term Of runtime efficiency:
- LSTM and Gru: 9-11ms/step
- BiLSTM and BiGRU: 15-17ms/step

=> The GRU is the best choice for most efficient runtime and high accuracy